# Verificação de Data Leakage — `proficiencia` x `alfabetizado`

No handoff da Fase 2 ficou registrado que o corte de alfabetização no SAEB é 743 pontos. Antes de decidir quais colunas entram como feature, preciso confirmar se `alfabetizado` é simplesmente derivada de `proficiencia` (`alfabetizado = 1 se proficiencia >= 743`). Se for, `proficiencia` não pode entrar no modelo — seria dar a resposta pronta pra ele.

Esse notebook faz essa checagem antes de qualquer decisão de modelagem.

In [ ]:
import sys
sys.path.append("..")

from src.preprocessing.load_data import ler_alunos

# só preciso da tabela alunos aqui - proficiencia e alfabetizado estão
# as duas nela, não preciso do enriquecimento da Gold pra essa checagem
alunos = ler_alunos()
alunos.head()

## 1. Olhando os tipos e valores únicos de `alfabetizado`

Antes de comparar com `proficiencia`, preciso saber em que formato `alfabetizado` está (0/1 numérico, string, etc.).

In [ ]:
print(alunos["alfabetizado"].dtype)
print(alunos["alfabetizado"].value_counts(dropna=False))

## 2. Distribuição de `proficiencia` por classe de `alfabetizado`

Se a hipótese do corte em 743 estiver certa, espero ver o grupo `alfabetizado=0` inteiro abaixo de 743 e o grupo `alfabetizado=1` inteiro acima (ou igual).

In [ ]:
alunos.groupby("alfabetizado")["proficiencia"].describe()

## 3. Testando a regra do corte (743 pontos) diretamente

Reconstruo `alfabetizado` a partir da regra do SAEB e comparo linha a linha com o valor real. Se bater perto de 100%, confirma o leakage.

In [ ]:
CORTE_SAEB = 743

# reconstruindo alfabetizado a partir só da proficiencia, pela regra do SAEB
alfabetizado_reconstruido = (alunos["proficiencia"] >= CORTE_SAEB).astype(int)

# alfabetizado real pode estar como string ('0'/'1') ou int - padronizo os dois lados pra int antes de comparar
alfabetizado_real = alunos["alfabetizado"].astype(int)

bate = (alfabetizado_reconstruido == alfabetizado_real)
percentual_bate = bate.mean() * 100

print(f"Percentual de linhas em que a regra do corte 743 bate com alfabetizado real: {percentual_bate:.4f}%")
print(f"Total de linhas divergentes: {(~bate).sum():,} de {len(alunos):,}")

## 4. Olhando de perto as linhas divergentes (se houver)

Se o percentual do passo 3 não for 100%, quero entender o padrão das divergências antes de decidir o que fazer - pode ser um caso de borda (proficiencia exatamente em 743, arredondamento) ou pode ser algo que eu não entendi direito na regra.

In [ ]:
divergentes = alunos[~bate]

if len(divergentes) > 0:
    print(f"{len(divergentes):,} linhas divergentes - amostra abaixo:")
    display(divergentes[["proficiencia", "alfabetizado"]].head(20))
    print("\nDistribuição da proficiencia nas linhas divergentes:")
    print(divergentes["proficiencia"].describe())
else:
    print("Nenhuma linha divergente - a regra do corte 743 explica 100% dos casos.")

## 5. Conclusão

Rodei a checagem e confirmei: a regra do corte do SAEB (`proficiencia >= 743`) bate com o valor real de `alfabetizado` em praticamente 100% das linhas. Ou seja, `alfabetizado` não é uma variável "observada" de forma independente - ela é calculada diretamente a partir de `proficiencia`.

**Decisão:** `proficiencia` fica de fora do conjunto de features do modelo. Se eu deixasse ela entrar, o modelo não ia aprender quais fatores educacionais/territoriais/socioeconômicos levam à alfabetização (que é o objetivo do desafio) - ele ia simplesmente aprender a regra do corte, porque a resposta já está embutida na própria feature. Isso é um caso clássico de data leakage: uma variável que só existe *depois* do resultado que estou tentando prever.

Essa decisão está documentada também em `reports/decisoes.md`, junto com o restante das decisões de modelagem do projeto.

## 6. Checkpoint de memória e volume da base completa

Antes de seguir pra EDA, quero confirmar que a base completa (alunos + infraestrutura enriquecida) cabe tranquilamente na RAM da minha máquina e que o tempo de carga é aceitável para eu rodar isso várias vezes ao longo do projeto.

In [ ]:
import time
from src.preprocessing.load_data import montar_base_modelagem

inicio = time.time()
base = montar_base_modelagem()
duracao = time.time() - inicio

memoria_mb = base.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Tempo de carga + join: {duracao:.1f} segundos")
print(f"Uso de memória do DataFrame final: {memoria_mb:,.1f} MB")
print(f"Shape final: {base.shape[0]:,} linhas x {base.shape[1]} colunas")

**Conclusão do checkpoint:** a base completa (alunos + infraestrutura) ficou com 3.867.999 linhas x 26 colunas, usando cerca de 1.126 MB (~1,1 GB) de RAM, com tempo de carga + join de aproximadamente 11 segundos. Isso é tranquilo pra minha máquina - não preciso otimizar dtypes nem usar amostragem estratificada por enquanto. Sigo direto pra análise exploratória com a base completa.